# ETL-25 · Validação pós-carga

- **Etapa:** ETL-25, Validação pós-carga
- **Responsável:** Pedro Wellington
- **Issue:** ETL-25
- **Entrada:** as 9 tabelas carregadas no banco (ETL-24)
- **Saída:** `sql/validacao/etl_checks.sql`

Confere o banco depois da carga: contagens por tabela, chaves estrangeiras sem correspondência, total de óbitos e três consultas de exemplo.

## 1. Setup

In [16]:
import os
from pathlib import Path
import pandas as pd

# Raiz do repositório: sobe de pasta em pasta até achar a pasta .git
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".git").exists())
os.chdir(RAIZ)   # roda a partir da raiz, para achar a pasta src/ e o arquivo .env

from src.conexao import obter_engine, testar_conexao

engine = obter_engine()
SAIDA_SQL = RAIZ / "sql" / "validacao" / "etl_checks.sql"   # arquivo com as consultas
SAIDA_SQL.parent.mkdir(parents=True, exist_ok=True)

CONSULTAS = {}   # nome -> texto SQL, para gravar no arquivo no final

def consultar(nome: str, sql: str) -> pd.DataFrame:
    """Roda a consulta no banco e guarda o texto para o arquivo etl_checks.sql."""
    CONSULTAS[nome] = sql.strip()
    return pd.read_sql(sql, engine)

_ = testar_conexao()

 Conexão estabelecida com sucesso!
 Banco de Dados : postgres
 Usuário         : postgres
 Versão SGDB     : PostgreSQL 17.11 on aarch64-unknown-linux-gnu...


## 2. Contagens por tabela

In [17]:
consultar('contagem_por_tabela', """
SELECT 'dim_tempo' AS tabela, COUNT(*) AS linhas FROM dim_tempo
UNION ALL SELECT 'dim_gravidade', COUNT(*) FROM dim_gravidade
UNION ALL SELECT 'dim_perfil_vitima', COUNT(*) FROM dim_perfil_vitima
UNION ALL SELECT 'dim_transporte', COUNT(*) FROM dim_transporte
UNION ALL SELECT 'dim_localidade', COUNT(*) FROM dim_localidade
UNION ALL SELECT 'dim_via', COUNT(*) FROM dim_via
UNION ALL SELECT 'dim_profissao', COUNT(*) FROM dim_profissao
UNION ALL SELECT 'dim_local_obito', COUNT(*) FROM dim_local_obito
UNION ALL SELECT 'fato_pessoa_sinistro', COUNT(*) FROM fato_pessoa_sinistro
""")

,tabela,linhas
0,dim_tempo,1121
1,dim_gravidade,6
2,dim_perfil_vitima,1500
3,dim_transporte,25
4,dim_localidade,647
5,dim_via,10
6,dim_profissao,561
7,dim_local_obito,5
8,fato_pessoa_sinistro,806977


## 3. Chaves estrangeiras sem correspondência

In [18]:
consultar('fk_sem_correspondencia', """
SELECT 'dim_tempo (data_sinistro)' AS chave, COUNT(*) AS orfaos
FROM fato_pessoa_sinistro f LEFT JOIN dim_tempo d ON f.sk_data_sinistro = d.sk_tempo
WHERE d.sk_tempo IS NULL
UNION ALL SELECT 'dim_tempo (data_obito)', COUNT(*)
FROM fato_pessoa_sinistro f LEFT JOIN dim_tempo d ON f.sk_data_obito = d.sk_tempo
WHERE d.sk_tempo IS NULL
UNION ALL SELECT 'dim_gravidade', COUNT(*)
FROM fato_pessoa_sinistro f LEFT JOIN dim_gravidade d ON f.sk_gravidade = d.sk_gravidade
WHERE d.sk_gravidade IS NULL
UNION ALL SELECT 'dim_perfil_vitima', COUNT(*)
FROM fato_pessoa_sinistro f LEFT JOIN dim_perfil_vitima d ON f.sk_perfil_vitima = d.sk_perfil_vitima
WHERE d.sk_perfil_vitima IS NULL
UNION ALL SELECT 'dim_transporte', COUNT(*)
FROM fato_pessoa_sinistro f LEFT JOIN dim_transporte d ON f.sk_transporte = d.sk_transporte
WHERE d.sk_transporte IS NULL
UNION ALL SELECT 'dim_localidade', COUNT(*)
FROM fato_pessoa_sinistro f LEFT JOIN dim_localidade d ON f.sk_localidade = d.sk_localidade
WHERE d.sk_localidade IS NULL
UNION ALL SELECT 'dim_via', COUNT(*)
FROM fato_pessoa_sinistro f LEFT JOIN dim_via d ON f.sk_via = d.sk_via
WHERE d.sk_via IS NULL
UNION ALL SELECT 'dim_profissao', COUNT(*)
FROM fato_pessoa_sinistro f LEFT JOIN dim_profissao d ON f.sk_profissao = d.sk_profissao
WHERE d.sk_profissao IS NULL
UNION ALL SELECT 'dim_local_obito', COUNT(*)
FROM fato_pessoa_sinistro f LEFT JOIN dim_local_obito d ON f.sk_local_obito = d.sk_local_obito
WHERE d.sk_local_obito IS NULL
""")

,chave,orfaos
0,dim_tempo (data_sinistro),0
1,dim_tempo (data_obito),0
2,dim_gravidade,0
3,dim_perfil_vitima,0
4,dim_transporte,0
5,dim_localidade,0
6,dim_via,0
7,dim_profissao,0
8,dim_local_obito,0


## 4. Total de óbitos

In [19]:
consultar('total_obitos', """
SELECT SUM(flag_obito) AS obitos, COUNT(*) AS vitimas
FROM fato_pessoa_sinistro
""")

,obitos,vitimas
0,15793,806977


## 5. Consultas de exemplo

Três consultas que juntam a fato com as dimensões, para conferir que os relacionamentos funcionam.

In [20]:
consultar('obitos_por_ano', """
SELECT t.ano, SUM(f.flag_obito) AS obitos, COUNT(*) AS vitimas
FROM fato_pessoa_sinistro f
JOIN dim_tempo t ON f.sk_data_sinistro = t.sk_tempo
GROUP BY t.ano
ORDER BY t.ano
""")

,ano,obitos,vitimas
0,2021,4858,260886
1,2022,5445,266128
2,2023,5490,279963


In [21]:
consultar('top10_municipios_obitos', """
SELECT l.municipio, SUM(f.flag_obito) AS obitos
FROM fato_pessoa_sinistro f
JOIN dim_localidade l ON f.sk_localidade = l.sk_localidade
GROUP BY l.municipio
ORDER BY obitos DESC
LIMIT 10
""")

,municipio,obitos
0,SAO PAULO,2525
1,CAMPINAS,445
2,GUARULHOS,423
3,SAO BERNARDO DO CAMPO,282
4,RIBEIRAO PRETO,268
5,SOROCABA,265
6,SAO JOSE DOS CAMPOS,218
7,JUNDIAI,211
8,MOGI DAS CRUZES,167
9,OSASCO,167


In [22]:
consultar('vitimas_por_sexo_e_tipo', """
SELECT p.sexo, tr.tipo_de_vitima, COUNT(*) AS vitimas, SUM(f.flag_obito) AS obitos
FROM fato_pessoa_sinistro f
JOIN dim_perfil_vitima p ON f.sk_perfil_vitima = p.sk_perfil_vitima
JOIN dim_transporte tr ON f.sk_transporte = tr.sk_transporte
GROUP BY p.sexo, tr.tipo_de_vitima
ORDER BY vitimas DESC
""")

,sexo,tipo_de_vitima,vitimas,obitos
0,MASCULINO,CONDUTOR,297070,9079
1,MASCULINO,PASSAGEIRO,182574,785
2,FEMININO,PASSAGEIRO,88746,969
3,FEMININO,CONDUTOR,79557,721
4,MASCULINO,NÃO INFORMADO,74699,500
5,FEMININO,NÃO INFORMADO,33999,110
6,MASCULINO,PEDESTRE,26104,2742
7,FEMININO,PEDESTRE,19862,821
8,NÃO INFORMADO,NÃO INFORMADO,4130,7
9,NÃO INFORMADO,PEDESTRE,207,31


## 6. Gravar o arquivo SQL

In [23]:
cabecalho = (
    "-- Consultas de validação pós-carga (ETL-25)\n"
    "-- Geradas pelo notebook notebooks/etl/10_validacao.ipynb\n\n"
)
blocos = [f"-- {nome}\n{sql};\n" for nome, sql in CONSULTAS.items()]

SAIDA_SQL.write_text(cabecalho + "\n".join(blocos), encoding="utf-8")
print(f"{len(CONSULTAS)} consultas gravadas em {SAIDA_SQL.relative_to(RAIZ)}")

6 consultas gravadas em sql\validacao\etl_checks.sql


## 7. Resumo

- As 9 tabelas estão no banco com as contagens esperadas: 8 dimensões e a fato com 806.977 linhas.
- Nenhuma chave estrangeira da fato ficou sem correspondência nas dimensões (0 em todas as 9 chaves).
- O total de óbitos na fato é 15.793, igual ao da base original.
- Foram escritas 3 consultas de exemplo juntando a fato com as dimensões.
- As consultas ficam salvas em `sql/validacao/etl_checks.sql`.